<style>
p { max-width: 80em;}
</style>

## <span style="color:#ce4356">1 </span>  - Bayesian thinking and building intuition

## Bayes' theorem

$$
\begin{aligned}
\underbrace{P(\text{Model}\,|\,\text{Data})}_{\textstyle\color{#888888}\text{posterior}}
\;&=\;
\frac{\overbrace{P(\text{Model})}^{\textstyle\color{#888888}\text{prior}}\;\overbrace{P(\text{Data}\,|\,\text{Model})}^{\textstyle\color{#888888}\text{likelihood}}}
     {\underbrace{P(\text{Data})}_{\textstyle\color{#888888}\text{evidence}}}
\;&=\;
\frac{P(\text{Model}_k)\;P(\text{Data}\,|\,\text{Model}_k)}
     {\underbrace{\sum_i P(\text{Model}_i)\;P(\text{Data}\,|\,\text{Model}_i)}_{\textstyle\color{#888888}\text{marginalised over every Model}}}
\end{aligned}
$$

An update process:

- **posterior** — how much credence should I give this model in the light of the new data?
- **prior** — how much did I believe it beforehand?
- **likelihood** — if the model were true, how probable is the data we got?
- **evidence** — how probable is the data under all the models together?  A normalisation factor.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from bayes_tools import bayes_table

## The Bayes table

One row per Model. $\text{prior} \times \text{likelihood}$ is the unnormalised posterior;
the **total of that column is the evidence** $P(\text{Data})$, and dividing by it gives
the posterior.

| Hypotheses | prior | likelihood | prior $\times$ likelihood | posterior |
|:---:|:---:|:---:|:---:|:---:|
| $\text{Model}_1$ | $P(\text{Model}_1)$ | $P(\text{Data} \mid \text{Model}_1)$ | $P(\text{Model}_1) \times P(\text{Data} \mid \text{Model}_1)$ | $P(\text{Model}_1 \mid \text{Data})$ |
| $\text{Model}_2$ | $P(\text{Model}_2)$ | $P(\text{Data} \mid \text{Model}_2)$ | $P(\text{Model}_2) \times P(\text{Data} \mid \text{Model}_2)$ | $P(\text{Model}_2 \mid \text{Data})$ |
| $\dots$ | $\dots$ | $\dots$ | $\dots$ | $\dots$ |
| Totals: | $1$ | | $P(\text{Data})$ | $1$ |


---
## <span style="color:#ce4356">Example 1.1 </span> — the medical test

- a rare disease 1% of population, $P(\text{sick}) = 0.01$
- false positives in 5% cases, $P(\text{positive}\,|\,\text{healthy}) = 0.05$
- the test never misses a real case, $P(\text{positive}\,|\,\text{sick}) = 1$

#### Patient tests positive, how do we update the belief about patient being sick? 

In [4]:
post = bayes_table(["sick", "healthy"],
                   prior=[0.01, 0.99],
                   likelihood=[1.0, 0.05])

Hypotheses,prior,likelihood,prior × likelihood,posterior
sick,0.01,1,0.01,0.1681
healthy,0.99,0.05,0.0495,0.8319
Totals:,1,,0.0595,1


### About 16% (!)

Cool 3Blue1Brown video about it: https://www.youtube.com/watch?v=lG4VkPoG3ko

---
## <span style="color:#ce4356">Example 1.2 </span>  — an alien signal

We detect a *perfect* signal — exactly what aliens would produce, so
$P(\text{Data}\,|\,\text{aliens}) = 1$. How much should we believe it?

- **aliens** — real aliens. $P(\text{aliens})$ is not 1 probably and not $10^{-10}$?
- **not aliens** — noise, terrestrial interference, an artefact of the pipeline, etc.

Say you can name **99** cases this signal could come from, each with some imperfection:
$P(\text{Data}\,|\,\text{not aliens}_i) = 0.8$ — and each *a priori* as plausible as the aliens.

So: 100 Models, each with a prior of 0.01.

| Hypotheses | prior | likelihood |
|:---:|:---:|:---:|
| aliens | 0.01 | 1 |
| noise | 0.01 | 0.8 |
| terrestrial interference | 0.01 | 0.8 | 
| glitch | 0.01 | 0.8 | 
| $\dots$ | $\dots$ | $\dots$ | 
| Totals: | $1$ | | 

In [5]:
post_A = bayes_table(["aliens", "not aliens"],
                     prior=[0.01, 0.99],
                     likelihood=[1.0, 0.8])

Hypotheses,prior,likelihood,prior × likelihood,posterior
aliens,0.01,1,0.01,0.01247
not aliens,0.99,0.8,0.792,0.9875
Totals:,1,,0.802,1


Why almost no change? The denominator:

$$P(\text{Data}) \;=\; \underbrace{1 \times 0.01}_{\text{aliens}} \;+\; \underbrace{99 \times 0.8 \times 0.01}_{\text{the 99 other cases}} \;=\; 0.802$$

Each case explains the signal a bit worse than aliens do — but there are 99 of them, and
they own 99% of the prior. So even a model that explains the data perfectly can end up with
a small posterior.

---
## <span style="color:#ce4356">Example 1.3 </span>  — which line is it?

A stellar spectrum shows one strong absorption
line near 3969 Å. Four candidate transitions:

| Model | $\lambda_{\rm rest}$ | strong in |
|:---|---:|:---|
| Ca II K | 3933.66 Å | cool stars |
| Ca II H | 3968.47 Å | cool stars |
| $\text{H}\epsilon$ | 3970.07 Å | hot stars |
| $\text{H}\delta$ | 4101.73 Å | hot stars |

### <span style="color:#ce4356">a) </span> Measurement 1 — the line centre, at low resolution

$\lambda_{\rm obs} = 3969.0 \pm 15$ Å, so $P(\lambda \mid \text{Model}) \;=\; \exp\!\left[-\tfrac{1}{2}\left(\frac{\lambda_{\rm obs} - \lambda_{\rm Model}}{\sigma_\lambda}\right)^{\!2}\right]$

In [6]:
names = ["Ca II K", "Ca II H", "Hε", "Hδ"]
lam0 = np.array([3933.66, 3968.47, 3970.07, 4101.73])
flat = np.full(4, 0.25)

lam_obs, sig_lam = 3969.0, 15.0
L_lam = np.exp(-0.5 * ((lam_obs - lam0) / sig_lam) ** 2)

post_lam = bayes_table(names, prior=flat, likelihood={"P(λ | Model)": L_lam})

Hypotheses,prior,P(λ | Model),prior × likelihood,posterior
Ca II K,0.25,0.06233,0.01558,0.03027
Ca II H,0.25,0.9994,0.2498,0.4853
Hε,0.25,0.9975,0.2494,0.4844
Hδ,0.25,9.945e-18,2.486e-18,4.83e-18
Totals:,1,,0.5148,1


$\text{H}\delta$ is excluded and Ca II K is down to 3%, but **Ca II H and $\text{H}\epsilon$ are 1.6 Å
apart** — nothing separates them at this resolution.

### <span style="color:#ce4356">b) </span> Measurement 2 — the colour of the star

B (blue) − V (visual) is a simple, powerful proxy for a star's surface temperature: cooler
stars are redder (larger B−V), hotter stars are bluer (smaller B−V).

We measure $B - V = 0.72 \pm 0.05$ for this star.

From stellar samples: stars whose spectra are Ca II-dominated (cool, G–K type) have
$B-V \approx 0.80 \pm 0.25$; stars whose spectra are Balmer-dominated (hot, A type) have
$B-V \approx 0.10 \pm 0.15$. We can model the second likelihood as a Gaussian population
around the mean:

$$P(B\!-\!V \mid \text{Model}) \;=\; \frac{1}{\sigma\sqrt{2\pi}}\exp\!\left[-\tfrac{1}{2}\left(\frac{(B-V)_{\rm obs} - (B-V)_{\rm Model}}{\sigma}\right)^{\!2}\right],
\qquad \sigma = \sqrt{\sigma_{\rm model}^2 + \sigma_{\rm obs}^2}$$

The two measurements are independent given the Model, so they combine as

$$P(\lambda, B\!-\!V \mid \text{Model}) = P(\lambda \mid \text{Model})\;P(B\!-\!V \mid \text{Model})$$

In [7]:
bv_obs = 0.72
sigma_obs = 0.05
bv_pred = np.array([0.80, 0.80, 0.10, 0.10])       # cool, cool, hot, hot
sigma_model = np.array([0.25, 0.25, 0.15, 0.15])   # intrinsic spread of each class

sigma = np.sqrt(sigma_model**2 + sigma_obs**2)
L_bv = (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-0.5 * ((bv_obs - bv_pred) / sigma) ** 2)

post_both = bayes_table(names, prior=flat,
                        likelihood={"P(λ | Model)": L_lam,
                                    "P(B-V | Model)": L_bv})

Hypotheses,prior,P(λ | Model),P(B-V | Model),prior × likelihood,posterior
Ca II K,0.25,0.06233,1.49,0.02321,0.05866
Ca II H,0.25,0.9994,1.49,0.3722,0.9406
Hε,0.25,0.9975,0.001156,0.0002883,0.0007288
Hδ,0.25,9.945e-18,0.001156,2.875e-21,7.266e-21
Totals:,1,,,0.3957,1


Alternatively, one can use the posterior after the first dataset as the prior for the second:

$$\underbrace{P(\lambda \mid M)\,P(M)}_{\textstyle\color{#888888}\text{posterior after }\lambda}
\;\equiv\; \underbrace{P'(M)}_{\textstyle\color{#888888}\text{new prior}}
\qquad\Longrightarrow\qquad
P(M \mid \lambda,\, B\!-\!V) \;\propto\; P(B\!-\!V \mid M)\;P'(M)$$